In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, resample
from scipy.optimize import curve_fit
from scipy.spatial.distance import cosine
from scipy.stats import skew, kurtosis
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
from google.colab import drive
import warnings

In [ ]:
# Change your paths here
FILTERED_INPUT_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/100Hz_4min_windows/BW_spline/'
OUTPUT_FILE = '/content/drive/MyDrive/2025_PPG_GLUC/Data/100Hz_ppg_features_4min_BW_spline_metadata.csv'
GLUCOSE_FILE = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Final Data/28_Extracted_Features/Other Attempts/valid_glucose.csv'

FS = 100

VALLEY_DISTANCE = 100 // 2  # min distance between valleys (0.5s at 100Hz)
SIMILARITY_THRESHOLD = 0.85
MIN_PULSE_LENGTH = 50  # Minimum samples for a valid pulse
MAX_PULSE_LENGTH = 1.5 * 100  # Maximum samples for a valid pulse (1.5s)

N_JOBS = -1
TEST_MODE = False  # Set to True to test on limited cases
TEST_CASES = 100

In [ ]:
import csv
import os
FS = 100
MIN_PULSE_LENGTH = 50  # Minimum samples for a valid pulse
MAX_PULSE_LENGTH = 1.5 * 100  # Maximum samples for a valid pulse (1.5s)

# @title
import traceback
def normalize_pulse(pulse):
    """Normalize pulse to [0, 1] range."""
    pulse_min = pulse.min()
    pulse_max = pulse.max()
    pulse_range = pulse_max - pulse_min

    if pulse_range < 1e-8:
        return None

    return (pulse - pulse_min) / pulse_range


def two_gaussian(t, H1, n1, W1, H2, n2, W2):
    """Two-Gaussian model for PPG pulse decomposition."""
    g1 = H1 * np.exp(-((t - n1) ** 2) / (2 * W1 ** 2 + 1e-8))
    g2 = H2 * np.exp(-((t - n2) ** 2) / (2 * W2 ** 2 + 1e-8))
    return g1 + g2


def fit_gaussian(pulse):
    """
    Fit two-Gaussian model to a normalized pulse (0-1 range, 50 samples).

    Returns: (H1, n1, W1, H2, n2, W2) or None if fitting fails
    """
    t = np.arange(len(pulse))
    n = len(pulse)

    # Find initial estimates from signal
    systolic_idx = np.argmax(pulse)
    systolic_height = pulse[systolic_idx]

    # Estimate diastolic region
    search_start = min(systolic_idx + 5, n - 5)
    search_end = min(systolic_idx + 35, n - 1)

    if search_end <= search_start:
        search_start = systolic_idx + 3
        search_end = n - 1

    search_region = pulse[search_start:search_end]

    # Find initial diastolic estimate
    if len(search_region) > 3:
        local_peaks, _ = find_peaks(search_region, distance=3)
        if len(local_peaks) > 0:
            peak_heights = search_region[local_peaks]
            best_peak = local_peaks[np.argmax(peak_heights)]
            diastolic_idx_init = search_start + best_peak
            diastolic_height_init = pulse[diastolic_idx_init]
        else:
            diastolic_idx_init = int(systolic_idx + 0.5 * (n - systolic_idx))
            diastolic_height_init = pulse[diastolic_idx_init]
    else:
        diastolic_idx_init = int(0.7 * n)
        diastolic_height_init = pulse[diastolic_idx_init]

    # Ensure diastolic is after systolic
    if diastolic_idx_init <= systolic_idx:
        diastolic_idx_init = min(systolic_idx + 10, n - 1)
        diastolic_height_init = pulse[diastolic_idx_init]

    # Initial parameter estimates
    p0 = [
        systolic_height,
        systolic_idx,
        5.0,
        max(diastolic_height_init, 0.1),
        diastolic_idx_init,
        5.0
    ]

    # Bounds
    bounds = (
        [0.5, 0, 2, 0.01, systolic_idx + 3, 2],
        [1.5, n * 0.5, 15, 0.8, n - 1, 15]
    )

    try:
        popt, _ = curve_fit(two_gaussian, t, pulse, p0=p0, bounds=bounds, maxfev=3000)
        H1, n1, W1, H2, n2, W2 = popt

        # Validate: n2 should be clearly after n1
        if n2 <= n1 + 3:
            return None

        return (H1, n1, W1, H2, n2, W2)

    except Exception:
        return None


def get_landmarks_from_gaussian(pulse, gaussian_params):
    """
    Extract landmarks using Gaussian parameters as guide.

    - idx_p1 = round(n1) → Systolic peak
    - idx_p2 = round(n2) → Diastolic peak
    - idx_notch = argmin between idx_p1 and idx_p2 (exclusive)
    """
    H1, n1, W1, H2, n2, W2 = gaussian_params
    n = len(pulse)

    # Systolic peak position from n1
    idx_p1 = int(np.clip(round(n1), 0, n - 1))

    # Diastolic peak position from n2
    idx_p2 = int(np.clip(round(n2), 0, n - 1))

    # Ensure idx_p2 > idx_p1
    if idx_p2 <= idx_p1:
        idx_p2 = min(idx_p1 + 5, n - 1)

    # Dicrotic notch: minimum btw sys and dia excluding endpoints
    if idx_p2 > idx_p1 + 2:
        search_segment = pulse[idx_p1 + 1 : idx_p2]  # Exclude both endpoints
        idx_notch = np.argmin(search_segment) + idx_p1 + 1
    else:
        # Peaks too close, estimate notch at midpoint
        idx_notch = (idx_p1 + idx_p2) // 2

    # Get values at landmarks
    val_p1 = pulse[idx_p1]
    val_p2 = pulse[idx_p2]
    val_notch = pulse[idx_notch]

    return {
        'idx_p1': idx_p1,
        'idx_p2': idx_p2,
        'idx_notch': idx_notch,
        'val_p1': val_p1,
        'val_p2': val_p2,
        'val_notch': val_notch
    }

def detect_valleys(signal, distance=FS*0.4):
    """Detect valleys (troughs) in the signal."""
    valleys, _ = find_peaks(-signal, distance=distance, prominence=0.1)
    return valleys

def extract_pulses_with_indices(signal, valleys):
    """
    Extract valley-to-valley segments with their original indices.

    Returns:
        pulses_normalized: List of normalized, resampled pulses
        pulse_metadata: List of dicts with valley_start, valley_end, original_length
    """
    pulses_normalized = []
    pulse_metadata = []

    for i in range(len(valleys) - 1):
        valley_start = valleys[i]
        valley_end = valleys[i + 1]
        original_length = valley_end - valley_start

        if MIN_PULSE_LENGTH <= original_length <= MAX_PULSE_LENGTH:
            pulse = signal[valley_start:valley_end]

            if np.any(np.isnan(pulse)):
                continue

            pulse_normalized = normalize_pulse(pulse)
            if pulse_normalized is None:
                continue

            pulses_normalized.append(pulse_normalized)
            pulse_metadata.append({
                'valley_start': valley_start,
                'valley_end': valley_end,
                'original_pulse_length': original_length
            })

    return pulses_normalized, pulse_metadata


def compute_template(pulses):
    """Compute template as mean of all pulses."""
    if len(pulses) == 0:
        return None
    return np.mean(pulses, axis=0)


def compute_similarities(pulses, template):
    """Compute cosine similarity for each pulse to template."""
    if template is None:
        return [0.0] * len(pulses)

    similarities = []
    for pulse in pulses:
        similarity = 1 - cosine(pulse, template)
        similarities.append(similarity)

    return similarities


def valid_fit(pulse, gaussian_params):
  """
  Check that H1 > H2 and notch > valley
  """
  H1, n1, W1, H2, n2, W2 = gaussian_params
  n = len(pulse)

  landmarks = get_landmarks_from_gaussian(pulse, gaussian_params)
  idx_p1 = landmarks['idx_p1']
  idx_p2 = landmarks['idx_p2']
  idx_notch = landmarks['idx_notch']

  valid_H1 = H1 > H2
  valid_notch = landmarks['idx_notch'] < landmarks['val_p2']

  if valid_H1 and valid_notch:
    return True
  else:
    return False

def extract_key_features(pulse, gaussian_params):
    """
    Extract all 28 features from a normalized pulse using Gaussian-guided landmarks.

    Returns: dict of 28 features + validity flags
    """
    H1, n1, W1, H2, n2, W2 = gaussian_params
    n = len(pulse)

    # Get landmarks from Gaussian parameters
    landmarks = get_landmarks_from_gaussian(pulse, gaussian_params)
    idx_p1 = landmarks['idx_p1']
    idx_p2 = landmarks['idx_p2']
    idx_notch = landmarks['idx_notch']

    # QUALITY FLAGS
    valid_h1_h2 = H1 > H2
    valid_notch = landmarks['val_notch'] < landmarks['val_p2']


    # # height features (3)
    highest_peak = landmarks['val_p1']
    dis_peak = landmarks['val_p2']
    notch = landmarks['val_notch']

    return {
        # Quality flags
        'valid_h1_h2': valid_h1_h2,
        'valid_notch': valid_notch,
        # Gaussian features (6)
        'H1': H1, 'H2': H2, 'n1': n1, 'n2': n2, 'W1': W1, 'W2': W2, 'idx_p1': idx_p1, 'idx_p2': idx_p2, 'idx_notch': idx_notch,
        # # Height features (3)
        'highest_peak': highest_peak, 'dis_peak': dis_peak, 'notch': notch
    }

def chunk_list(lst, window_size=15, step=14):
    return [
        lst[i:i + window_size:step]
        for i in range(0, len(lst) - window_size + 1, step)
    ]

def process_single_file(filepath, glucose_value):
    """
    Process a single 16-min filtered PPG file.
    Returns list of feature dictionaries (one per valid pulse).
    """
    try:
        signal = np.load(filepath)
        valleys = detect_valleys(signal)

        if len(valleys) < 2:
            return []

        # Extract pulses with their original indices
        pulses, pulse_metadata = extract_pulses_with_indices(signal, valleys)

        if len(pulses) == 0:
            return []

        # Parse filename
        filename = Path(filepath).stem
        parts = filename.split('_')
        caseid = parts[1]
        glucose_time = parts[3]

        # Extract features for each valid pulse
        results = []

        for seg_idx, pulse in enumerate(pulses):
            meta = pulse_metadata[seg_idx]

            # Fit Gaussian
            gaussian_params = fit_gaussian(pulse)

            if gaussian_params is None:
                continue

            # Extract features (Not required here)
            features = extract_key_features(pulse, gaussian_params)

            # Add metadata
            features['caseid'] = caseid
            features['glucose_time'] = glucose_time
            features['glucose_value'] = glucose_value
            features['segment_index'] = seg_idx
            features['total_segments'] = len(pulses)
            features['valley_start'] = meta['valley_start']
            features['valley_end'] = meta['valley_end']
            features['original_pulse_length'] = meta['original_pulse_length']


            results.append(features)

        return results

    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        traceback.print_exc()
        return []

In [ ]:
def main():
    """Main function to process all files and create feature CSV."""
    print("=" * 70)
    print("PPG Feature Extraction v5 - 17 Columns")
    print("Gaussian-guided landmarks + valley indices + quality flags")
    print("=" * 70)

    # Load glucose data
    print("\nLoading glucose data...")
    glucose_df = pd.read_csv(GLUCOSE_FILE)
    print(f"  Glucose records: {len(glucose_df)}")

    # Create lookup dictionary
    glucose_lookup = {}
    for _, row in glucose_df.iterrows():
        key = (str(int(row['caseid'])), str(int(row['dt'])))
        glucose_lookup[key] = row['result']

    print("Glucose lookup: ", glucose_lookup)

    # Get list of filtered files
    print("\nScanning input directory...")
    input_path = Path(FILTERED_INPUT_PATH)
    files = list(input_path.glob('*.npy'))
    print(f"  Found {len(files)} filtered files")

    if TEST_MODE:
        files = files[:TEST_CASES]
        print(f"  TEST MODE: Processing only {TEST_CASES} files")

    # Match files to glucose values
    file_glucose_pairs = []
    for f in files:
        filename = f.stem
        parts = filename.split('_')
        caseid = parts[1]
        glucose_time = parts[3]
        key = (caseid, glucose_time)

        if key in glucose_lookup:
            file_glucose_pairs.append((str(f), glucose_lookup[key]))
        else:
            print(f"  Warning: No glucose match for {filename}")

    print(f"  Matched files: {len(file_glucose_pairs)}")

    # Process files in parallel
    print("\nExtracting features...")
    results = Parallel(n_jobs=-1)(
        delayed(process_single_file)(filepath, glucose_val)
        for filepath, glucose_val in tqdm(file_glucose_pairs, desc="Processing")
    )
    print("Results: ", results)

    all_features = []
    for file_results in results:
        all_features.extend(file_results)

    print(f"\nTotal segments extracted: {len(all_features)}")

    # Create DataFrame
    if len(all_features) > 0:
        df = pd.DataFrame(all_features)

        # Define column order (17 columns)
        metadata_cols = [
            'caseid', 'glucose_time', 'glucose_value',
            'segment_index', 'total_segments',
            'valley_start', 'valley_end', 'original_pulse_length'
        ]
        # quality_cols = ['cosine_similarity', 'valid_h1_h2', 'valid_notch']
        quality_cols = ['valid_h1_h2', 'valid_notch']
        gaussian_cols = ['H1', 'H2', 'n1', 'n2', 'W1', 'W2']

        feature_cols = gaussian_cols
        all_cols = metadata_cols + quality_cols + feature_cols

        df = df[all_cols]

        # Save to CSV
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\nSaved to: {OUTPUT_FILE}")
        print(f"  Shape: {df.shape} (rows, columns)")

        # STATISTICS
        print("\n" + "=" * 70)
        print("STATISTICS")
        print("=" * 70)

        # Quality flag distribution
        print("\nQuality Flags:")
        valid_h1h2_count = df['valid_h1_h2'].sum()
        valid_notch_count = df['valid_notch'].sum()
        both_valid = ((df['valid_h1_h2']) & (df['valid_notch'])).sum()

        print(f"  valid_h1_h2 = True:  {valid_h1h2_count:,} ({100*valid_h1h2_count/len(df):.1f}%)")
        print(f"  valid_notch = True:  {valid_notch_count:,} ({100*valid_notch_count/len(df):.1f}%)")
        print(f"  Both valid:          {both_valid:,} ({100*both_valid/len(df):.1f}%)")


        # Feature statistics
        print(f"\nFeature Ranges:")

        for col in feature_cols:
            print(f"  {col}: [{df[col].min():.4f}, {df[col].max():.4f}]")

        # NaN check
        nan_counts = df[feature_cols].isna().sum()
        total_nan = nan_counts.sum()
        print(f"\nNaN Values: {total_nan}")
        if total_nan > 0:
            for col, count in nan_counts[nan_counts > 0].items():
                print(f"  {col}: {count}")

        # Sample rows
        print("\n" + "=" * 70)
        print("SAMPLE ROWS (first 3)")
        print("=" * 70)
        print(df.head(3).to_string())

    else:
        print("\nNo features extracted!")

    print("\n" + "=" * 70)
    print("Done!")
    print("=" * 70)


if __name__ == "__main__":
    main()

## Binning into bins of 15

Here we bin the individual pulses obtained above into bins of 15.

In [ ]:
import pandas as pd
import numpy as np

def make_valid_pulse_bins(df, window_size=15, step=15):
    """
    Returns one row per bin:
      caseid, glucose_time, glucose_value, win_id,
      pulse_starts (list), pulse_ends (list), segment_indices (list)
    """
    df = df.copy()

    # filter valid pulses
    df = df[(df["valid_h1_h2"] == True) & (df["valid_notch"] == True) & (df["original_pulse_length"] <= 100)]

    out = []
    for (caseid, glucose_time), g in df.groupby(["caseid", "glucose_time"], sort=False):
        g = g.sort_values(["segment_index", "valley_start"]).reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        win_id = 0
        for start in range(0, n - window_size + 1, step):
            chunk = g.iloc[start:start + window_size]

            out.append({
                "caseid": caseid,
                "glucose_time": glucose_time,
                "glucose_value": float(chunk["glucose_value"].iloc[0]),
                "win_id": win_id,

                # store pulse boundaries so you can extract later
                "pulse_starts": chunk["valley_start"].astype(int).tolist(),
                "pulse_ends": chunk["valley_end"].astype(int).tolist(),
                "segment_indices": chunk["segment_index"].astype(int).tolist(),
                "n_pulses": window_size
            })
            win_id += 1

    return pd.DataFrame(out)
